# Validacion del proxy de deterioro anticipado
Este notebook valida la calidad estadistica y temporal del proxy antes de usarlo en modelos. No valida la verdad de campo: esa requiere eventos de congestion documentados.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_DIR = PROJECT_ROOT / 'data' / '04_model_output' / 'precolapso'
SOURCE_PATHS = {
    'metr_la': PROJECT_ROOT / 'data' / '01_processed' / 'external' / 'metr_la.parquet',
    'pems_bay': PROJECT_ROOT / 'data' / '01_processed' / 'external' / 'pems_bay.parquet',
}
HORIZONS = [5, 10, 15, 20, 30, 60]
THRESHOLDS = [0.10, 0.15, 0.20, 0.25, 0.30]
MIN_TEST_EVENTS = 10
print('fuentes=', list(SOURCE_PATHS), 'horizontes=', HORIZONS, 'umbrales=', THRESHOLDS)

fuentes= ['metr_la', 'pems_bay'] horizontes= [5, 10, 15, 20, 30, 60] umbrales= [0.1, 0.15, 0.2, 0.25, 0.3]


In [2]:
def load_speed_series(source, path):
    frame = pd.read_parquet(path)
    frame['timestamp'] = pd.to_datetime(frame['timestamp'], errors='coerce')
    sensors = [column for column in frame.columns if column != 'timestamp']
    result = pd.DataFrame({'timestamp': frame['timestamp'], 'speed_median': frame[sensors].apply(pd.to_numeric, errors='coerce').median(axis=1)})
    return result.dropna().sort_values('timestamp').reset_index(drop=True)

series_by_source = {source: load_speed_series(source, path) for source, path in SOURCE_PATHS.items()}
cadence_rows = []
for source, frame in series_by_source.items():
    deltas = frame['timestamp'].diff().dropna().dt.total_seconds().div(60)
    cadence_rows.append({'source': source, 'filas': len(frame), 'intervalo_mediano_min': deltas.median(), 'porcentaje_intervalos_5min': float((deltas == 5).mean() * 100), 'huecos_mayores_5min': int((deltas > 5).sum()), 'timestamp_invalido': int(frame['timestamp'].isna().sum())})
cadence = pd.DataFrame(cadence_rows)
display(cadence)

,source,filas,intervalo_mediano_min,porcentaje_intervalos_5min,huecos_mayores_5min,timestamp_invalido
0,metr_la,34272,5.0,100.000000,0,0
1,pems_bay,52116,5.0,99.998081,1,0


In [4]:
def label_quality(source, source_frame, horizon, threshold):
    frame = source_frame.copy()
    steps = horizon // 5
    frame['baseline_past'] = frame['speed_median'].shift(1).rolling(3, min_periods=3).median()
    frame['future_speed'] = frame['speed_median'].shift(-steps)
    frame['future_speed_next'] = frame['speed_median'].shift(-(steps + 1))
    frame['future_timestamp'] = frame['timestamp'].shift(-steps)
    frame['precolapso'] = (frame['future_speed'] <= frame['baseline_past'] * (1 - threshold)).astype(int)
    frame['precolapso_persistente'] = ((frame['future_speed'] <= frame['baseline_past'] * (1 - threshold)) & (frame['future_speed_next'] <= frame['baseline_past'] * (1 - threshold))).astype(int)
    frame = frame.dropna(subset=['baseline_past', 'future_speed', 'future_speed_next', 'future_timestamp']).copy()
    n = len(frame)
    train = frame.iloc[:int(n * 0.60)]
    validation = frame.iloc[int(n * 0.60):int(n * 0.80)]
    test = frame.iloc[int(n * 0.80):]
    all_events = int(frame['precolapso'].sum())
    persistent_events = int(frame['precolapso_persistente'].sum())
    persistence_rate = persistent_events / all_events if all_events else np.nan
    return {'source': source, 'horizon_minutes': horizon, 'threshold': threshold, 'filas': n, 'positivos_total': all_events, 'positivos_train': int(train['precolapso'].sum()), 'positivos_validation': int(validation['precolapso'].sum()), 'positivos_test': int(test['precolapso'].sum()), 'tasa_evento': float(frame['precolapso'].mean()), 'persistencia_evento_2_intervalos': persistence_rate, 'clases_train': int(train['precolapso'].nunique()), 'clases_validation': int(validation['precolapso'].nunique()), 'clases_test': int(test['precolapso'].nunique()), 'sin_fuga_temporal': bool((frame['timestamp'] < frame['future_timestamp']).all()), 'estado': 'usable_para_experimento' if train['precolapso'].nunique() == 2 and validation['precolapso'].nunique() == 2 and test['precolapso'].nunique() == 2 and int(test['precolapso'].sum()) >= MIN_TEST_EVENTS else 'insuficiente_evidencia'}

quality_rows = []
for source, frame in series_by_source.items():
    for threshold in THRESHOLDS:
        for horizon in HORIZONS:
            quality_rows.append(label_quality(source, frame, horizon, threshold))
label_quality_df = pd.DataFrame(quality_rows)
display(label_quality_df.head())
label_quality_df.to_csv(OUTPUT_DIR / 'calidad_etiqueta_precolapso.csv', index=False)

,source,horizon_minutes,threshold,filas,positivos_total,positivos_train,positivos_validation,positivos_test,tasa_evento,persistencia_evento_2_intervalos,clases_train,clases_validation,clases_test,sin_fuga_temporal,estado
0,metr_la,5,0.1,34267,2366,1379,255,732,0.069046,0.932798,2,2,2,True,usable_para_experimento
1,metr_la,10,0.1,34266,2475,1428,286,761,0.072229,0.931313,2,2,2,True,usable_para_experimento
2,metr_la,15,0.1,34265,2593,1489,315,789,0.075675,0.924026,2,2,2,True,usable_para_experimento
3,metr_la,20,0.1,34264,2716,1541,355,820,0.079267,0.920103,2,2,2,True,usable_para_experimento
4,metr_la,30,0.1,34262,2924,1634,423,867,0.085342,0.926813,2,2,2,True,usable_para_experimento


In [5]:
usable = label_quality_df[label_quality_df['estado'] == 'usable_para_experimento']
summary = {
    'status': 'etiqueta_exploratoria_validada_para_subconjunto' if not usable.empty else 'etiqueta_exploratoria_no_apta_para_modelado_general',
    'fuentes_con_combinacion_valida': sorted(usable['source'].unique().tolist()),
    'combinaciones_validas': int(len(usable)),
    'horizontes_evaluados': HORIZONS,
    'umbrales_evaluados': THRESHOLDS,
    'criterio_minimo_eventos_test': MIN_TEST_EVENTS,
    'nota': 'La validacion estadistica no sustituye etiquetas de eventos reales ni validacion local.',
}
(OUTPUT_DIR / 'validacion_etiqueta_precolapso.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
display(label_quality_df.groupby(['source', 'estado'], as_index=False).agg(combinaciones=('horizon_minutes', 'size'), max_positivos_test=('positivos_test', 'max')))
print(json.dumps(summary, indent=2, ensure_ascii=False))

,source,estado,combinaciones,max_positivos_test
0,metr_la,usable_para_experimento,30,974
1,pems_bay,insuficiente_evidencia,28,7
2,pems_bay,usable_para_experimento,2,20


{
  "status": "etiqueta_exploratoria_validada_para_subconjunto",
  "fuentes_con_combinacion_valida": [
    "metr_la",
    "pems_bay"
  ],
  "combinaciones_validas": 32,
  "horizontes_evaluados": [
    5,
    10,
    15,
    20,
    30,
    60
  ],
  "umbrales_evaluados": [
    0.1,
    0.15,
    0.2,
    0.25,
    0.3
  ],
  "criterio_minimo_eventos_test": 10,
  "nota": "La validacion estadistica no sustituye etiquetas de eventos reales ni validacion local."
}


## Criterio de interpretacion
Una etiqueta puede pasar a modelado solo cuando tiene dos clases en train, validation y test, al menos diez positivos en test y una cadencia temporal coherente. Si una fuente no cumple, se reporta como insuficiente y no se fuerza al modelo internacional.